# 03 - Data Cleaning & Feature Engineering (Quality Side)

**Stage 3 of the project, continued.**

This notebook cleans the eight quality-control fact tables (cap, bottle,
ink x variable/attribute/disposition), links every inspection record back
to its production `LotId`, and pre-computes the SPC (Statistical Process
Control) building blocks: X-bar/R control limits, Cp/Cpk/Pp/Ppk process
capability, and AQL attribute KPIs (p, DPU, DPMO). It also derives six
extra "clean" dimension tables (`dim_machine`, `dim_mold`, `dim_operator`,
`dim_bottle`, `dim_cap`, `dim_ink`) from the fact data, for a proper
star-schema in MySQL.

### Why some KPIs are columns here, and others are left to the BI layer
- **Per-row-computable, per-group-constant KPIs** (control limits, Cp/Cpk,
  DPU/DPMO) *are* pre-computed here, because they only need the row's own
  group (Characteristic x Machine x Mold/Product) and are cheap to
  broadcast back onto every row with a `groupby().transform()`.
- **True aggregates over many rows/tables** (FPY, Pareto of defects,
  MTBF/MTTR, inspector bias ranking, ...) are *not* stored as fact-table
  columns -- they belong in a view/query/notebook, computed at report
  time. Stage 4 and 5 notebooks compute these.


In [1]:
import sys
sys.path.insert(0, '.')
import pandas as pd
import numpy as np
import etl_lib as etl

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 180)

RAW = '../../datasets/raw'
DIM = '../../datasets/dim'
PROCESSED = '../../datasets/processed'


In [2]:
# fact_production_processed already carries the authoritative LotId per work
# order (built in notebook 02) -- every QC table links to it by joining on
# `WorkOrder`, which is a globally unique key across the whole plant in this
# dataset.
prod = pd.read_csv(f'{PROCESSED}/fact_production_processed.csv', encoding='utf-8-sig')
wo_to_lot = prod.set_index('WorkOrder')['LotId']

def clean_and_link(name, cat_cols=()):
    df = pd.read_csv(f'{RAW}/{name}.csv', encoding='utf-8-sig')
    df = etl.normalize_placeholder_nulls(df)
    df = etl.strip_whitespace(df)
    df = etl.standardize_categories(df, list(cat_cols))
    df, n_dupes = etl.drop_exact_duplicates(df)
    df['LotId'] = df['WorkOrder'].map(wo_to_lot)
    match_rate = df['LotId'].notna().mean()
    print(f"[{name}] removed {n_dupes} dupes, {len(df):,} rows, LotId match rate {match_rate:.1%}")
    return df


## 3.8 Cap (Injection Molding) quality tables


In [3]:
cap_var = clean_and_link('fact_cap_inspection_variable_cq_raw', ['Shift'])
cap_attr = clean_and_link('fact_cap_attribute_inspection_cq_raw', ['Shift'])
cap_disp = clean_and_link('fact_cap_disposition_lot_cq_raw', ['Shift'])


[fact_cap_inspection_variable_cq_raw] removed 0 dupes, 83,947 rows, LotId match rate 100.0%


[fact_cap_attribute_inspection_cq_raw] removed 0 dupes, 9,924 rows, LotId match rate 100.0%
[fact_cap_disposition_lot_cq_raw] removed 0 dupes, 1,654 rows, LotId match rate 100.0%


### X-bar / R control limits and Cp/Cpk

Grouped by **Characteristic x Machine x Mold x Product** -- the natural
"one process stream" grain for a control chart. Subgroup size for caps is
n=10 (`M1`...`M10`), so we use the A2/D3/D4 constants for n=10
(Montgomery's Table 6.1).


In [4]:
GROUP_COLS_CAP = ['Characteristic', 'MachineId', 'MoldId', 'CapId']
cap_var = etl.add_control_limits(cap_var, GROUP_COLS_CAP, subgroup_n=10)
cap_var = etl.add_process_capability(cap_var, GROUP_COLS_CAP, subgroup_n=10)

cap_var[cap_var['Characteristic'] == 'Weight'][
    ['MachineId', 'MoldId', 'XBarCL', 'XBarUCL', 'XBarLCL', 'Cp', 'Cpk', 'Pp', 'Ppk']
].drop_duplicates(['MachineId', 'MoldId']).head(6)


,MachineId,MoldId,XBarCL,XBarUCL,XBarLCL,Cp,Cpk,Pp,Ppk
0,IM-001,M-INJ-001,12.052597,12.381308,11.723885,1.682371,1.679875,1.740660,1.738077
78,IM-002,M-INJ-002,10.341420,10.675006,10.007834,1.657785,1.649657,1.820154,1.811230
115,IM-003,M-INJ-003,11.499366,11.818639,11.180093,1.682614,1.681987,1.619440,1.618836
181,IM-004,M-INJ-004,16.694418,17.149165,16.239672,1.667784,1.663906,2.041531,2.036783
234,IM-005,M-INJ-008,19.309286,19.920713,18.697859,1.653877,1.649078,1.912841,1.907291
268,IM-005,M-INJ-005,15.556704,16.005781,15.107627,1.653657,1.648939,1.977849,1.972207


In [5]:
cap_attr = etl.add_attribute_kpis(cap_attr)
cap_attr[['Characteristic', 'Class', 'DefectsFound', 'SampleSize',
          'DefectRateP', 'DPMO', 'LotDecision']].head()


,Characteristic,Class,DefectsFound,SampleSize,DefectRateP,DPMO,LotDecision
0,Thread,Critical,0,315,0.000000,0.000000,Approved
1,Sealing,Critical,0,315,0.000000,0.000000,Approved
2,Flash,Major,7,315,0.022222,22222.222222,Rejected
3,Short Shot,Major,0,315,0.000000,0.000000,Approved
4,Stain,Minor,0,315,0.000000,0.000000,Approved


## 3.9 Bottle (Blow Molding) quality tables

Subgroup size for bottles is n=5, so we use the n=5 SPC constants instead.


In [6]:
bottle_var = clean_and_link('fact_bottle_inspection_variables_cq_raw')
bottle_attr = clean_and_link('fact_bottle_attribute_inspection_cq_raw')
bottle_disp = clean_and_link('fact_bottle_disposition_lot_cq_raw', ['Shift'])

GROUP_COLS_BOTTLE = ['Characteristic', 'MachineId', 'MoldId', 'BottleId']
bottle_var = etl.add_control_limits(bottle_var, GROUP_COLS_BOTTLE, subgroup_n=5)
bottle_var = etl.add_process_capability(bottle_var, GROUP_COLS_BOTTLE, subgroup_n=5)
bottle_attr = etl.add_attribute_kpis(bottle_attr)

bottle_var[bottle_var['Characteristic'] == 'Weight'][
    ['MachineId', 'MoldId', 'Cp', 'Cpk']].drop_duplicates(['MachineId', 'MoldId']).head(6)


[fact_bottle_inspection_variables_cq_raw] removed 0 dupes, 124,580 rows, LotId match rate 100.0%


[fact_bottle_attribute_inspection_cq_raw] removed 0 dupes, 17,520 rows, LotId match rate 100.0%
[fact_bottle_disposition_lot_cq_raw] removed 0 dupes, 2,190 rows, LotId match rate 100.0%


,MachineId,MoldId,Cp,Cpk
0,ISBM-001,M-SOP-007,1.691777,1.669219
28,ISBM-001,M-SOP-001,1.663060,1.652927
111,ISBM-002,M-SOP-008,1.648999,1.637570
272,ISBM-003,M-SOP-009,1.722986,1.715924
387,ISBM-004,M-SOP-029,1.664006,1.645975
410,ISBM-004,M-SOP-026,1.666969,1.662863


## 3.10 Ink (Screen Printing) quality tables

Screen-printing quality control in this plant is 100% attribute-based (no
dimensional/variable measurements are taken -- registration, coverage,
color, adhesion, etc. are all pass/fail visual or gauge checks), which is
why there is no `fact_ink_inspection_variable_cq` table and no control
chart step here.


In [7]:
ink_attr = clean_and_link('fact_ink_attribute_inspection_cq_raw', ['Shift'])
ink_disp = clean_and_link('fact_ink_disposition_lot_cq_raw', ['Shift'])
ink_attr = etl.add_attribute_kpis(ink_attr)
ink_attr[['Characteristic', 'ColorCount', 'DefectsFound', 'DefectRateP', 'LotDecision']].head()


[fact_ink_attribute_inspection_cq_raw] removed 0 dupes, 4,456 rows, LotId match rate 100.0%
[fact_ink_disposition_lot_cq_raw] removed 0 dupes, 557 rows, LotId match rate 100.0%


,Characteristic,ColorCount,DefectsFound,DefectRateP,LotDecision
0,Registration,2,0,0.000000,Approved
1,Centering,2,1,0.003175,Approved
2,Coverage,2,0,0.000000,Approved
3,Color,2,0,0.000000,Approved
4,Smudge,2,0,0.000000,Approved


In [8]:
outputs = {
    'fact_cap_inspection_variable_cq_processed': cap_var,
    'fact_cap_attribute_inspection_cq_processed': cap_attr,
    'fact_cap_disposition_lot_cq_processed': cap_disp,
    'fact_bottle_inspection_variables_cq_processed': bottle_var,
    'fact_bottle_attribute_inspection_cq_processed': bottle_attr,
    'fact_bottle_disposition_lot_cq_processed': bottle_disp,
    'fact_ink_attribute_inspection_cq_processed': ink_attr,
    'fact_ink_disposition_lot_cq_processed': ink_disp,
}
for name, df in outputs.items():
    df.to_csv(f'{PROCESSED}/{name}.csv', encoding='utf-8-sig', index=False)
print("Quality-control processed files written:", list(outputs.keys()))


Quality-control processed files written: ['fact_cap_inspection_variable_cq_processed', 'fact_cap_attribute_inspection_cq_processed', 'fact_cap_disposition_lot_cq_processed', 'fact_bottle_inspection_variables_cq_processed', 'fact_bottle_attribute_inspection_cq_processed', 'fact_bottle_disposition_lot_cq_processed', 'fact_ink_attribute_inspection_cq_processed', 'fact_ink_disposition_lot_cq_processed']


## 3.11 Derived dimension tables

A proper star schema needs clean, deduplicated dimension tables rather
than repeating machine/mold/operator attributes on every fact row. We
derive six of them directly from the now-clean fact data:

- **`dim_machine`** -- one row per physical machine, with its process
- **`dim_mold`** -- one row per physical mold/tool, with capacity from
  `dim_machine_setup` where available
- **`dim_operator`** -- everyone who appears as a production operator or a
  maintenance technician, with their role
- **`dim_cap`** / **`dim_bottle`** / **`dim_ink`** -- product-level master
  data for each of the three decorated/molded families (bottle
  dimensional targets are synthesized here since no such master table
  existed upstream -- see the note in the previous project stage)


In [9]:
machine_setup = pd.read_csv(f'{DIM}/dim_machine_setup.csv', encoding='utf-8-sig')
cap_raw = pd.read_csv(f'{DIM}/dim_cap.csv', encoding='utf-8-sig')
masterbatch = pd.read_csv(f'{DIM}/dim_masterbatch.csv', encoding='utf-8-sig')

dim_machine = (prod[['MachineId', 'Process']].drop_duplicates()
               .sort_values(['Process', 'MachineId']).reset_index(drop=True))
dim_machine['MachineNumber'] = dim_machine['MachineId'].str.extract(r'(\d+)$').astype(int)

dim_mold = (prod[['ToolId', 'MachineId', 'Process']].drop_duplicates()
            .rename(columns={'ToolId': 'MoldId'}))
dim_mold = dim_mold.merge(
    machine_setup[['MoldId', 'Cavities', 'RatedCapacityPerHour', 'CyclesPerHour', 'IdealCycleTimeSec']],
    on='MoldId', how='left')

downtime = pd.read_csv(f'{PROCESSED}/fact_downtime_processed.csv', encoding='utf-8-sig')
ops = pd.concat([
    prod[['OperatorId', 'Process']].rename(columns={'OperatorId': 'PersonId'}).assign(Role='Operator'),
    downtime.loc[downtime['MaintenanceTechnician'] != 'N/A (Operations)',
                 ['MaintenanceTechnician', 'MaintenanceTeam']]
        .rename(columns={'MaintenanceTechnician': 'PersonId', 'MaintenanceTeam': 'Process'}).assign(Role='Maintenance'),
], ignore_index=True).dropna(subset=['PersonId'])
dim_operator = ops.drop_duplicates(subset=['PersonId', 'Role']).reset_index(drop=True)

dim_bottle = masterbatch[masterbatch['ProductType'] == 'Bottle'].copy()
import re
dim_bottle['VolumeMl'] = dim_bottle['ProductId'].apply(lambda p: int(re.search(r'(\d+)$', str(p)).group(1)))

screen_printing = prod[prod['Process'] == 'Screen Printing']
dim_ink = screen_printing[['ToolId', 'ProductId']].drop_duplicates().rename(columns={'ToolId': 'PrintToolId'})
dim_ink['ColorCount'] = dim_ink['PrintToolId'].str.extract(r'(\d+)C$').astype(float)

dim_machine.to_csv(f'{PROCESSED}/dim_machine.csv', encoding='utf-8-sig', index=False)
dim_mold.to_csv(f'{PROCESSED}/dim_mold.csv', encoding='utf-8-sig', index=False)
dim_operator.to_csv(f'{PROCESSED}/dim_operator.csv', encoding='utf-8-sig', index=False)
cap_raw.to_csv(f'{PROCESSED}/dim_cap.csv', encoding='utf-8-sig', index=False)
dim_bottle.to_csv(f'{PROCESSED}/dim_bottle.csv', encoding='utf-8-sig', index=False)
dim_ink.to_csv(f'{PROCESSED}/dim_ink.csv', encoding='utf-8-sig', index=False)

for n, d in {'dim_machine': dim_machine, 'dim_mold': dim_mold, 'dim_operator': dim_operator,
             'dim_cap': cap_raw, 'dim_bottle': dim_bottle, 'dim_ink': dim_ink}.items():
    print(f"{n}: {d.shape}")


dim_machine: (18, 3)
dim_mold: (215, 7)
dim_operator: (18, 3)
dim_cap: (31, 15)
dim_bottle: (101, 10)
dim_ink: (99, 3)


## 3.12 Hiding columns that don't help the analysis

`dim_bottle` inherits every column from `dim_masterbatch` (colorant
dosage details relevant to the *masterbatch* dimension, not to a bottle's
own physical spec), which is more than a bottle-quality analysis needs.
We keep the full table in `datasets/processed/` for traceability (it's
cheap, one row per product), but any notebook or Power BI report working
with bottle *specifications* should select only the physical columns:


In [10]:
bottle_spec_cols = ['ProductId', 'VolumeMl', 'BaseMaterial', 'MoldId']
dim_bottle[bottle_spec_cols].head()


,ProductId,VolumeMl,BaseMaterial,MoldId
0,FR-001-HDPE-300,300,HDPE,M-SOP-001
1,FR-002-HDPE-PCR-300,300,HDPE-PCR,M-SOP-002
2,FR-001-HDPE-PCR-300,300,HDPE-PCR,M-SOP-001
3,FR-001-rPET-300,300,rPET,M-SOP-001
4,FR-003-rPET-300,300,rPET,M-SOP-003
